# Benchmark MOPEP entre runtimes

Análise offline de qualidade, revisão e performance por runtime, bucket e turno. O notebook não envia requisições.

In [ ]:
from pathlib import Path
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display

DATASET_DIR = Path(os.environ.get('MOPEP_PERFORMANCE_DATASET_DIR', '../data/mopep-runtime-comparison')).resolve()
quality = pd.read_csv(DATASET_DIR / 'quality-by-request.csv')
summary = pd.read_csv(DATASET_DIR / 'quality-summary.csv')
deltas = pd.read_csv(DATASET_DIR / 'review-deltas.csv')
print(DATASET_DIR)

## Qualidade por runtime, perfil, bucket e turno

In [ ]:
display(summary.sort_values(['workload_profile', 'bucket', 'runtime', 'turn_index']))

## Custo de inferência por bucket

In [ ]:
metrics = [column for column in ['time_to_first_token_ms', 'end_to_end_latency_seconds', 'decode_tokens_per_second'] if column in quality]
fig, axes = plt.subplots(1, max(1, len(metrics)), figsize=(6 * max(1, len(metrics)), 5))
axes = [axes] if len(metrics) == 1 else axes
for axis, metric in zip(axes, metrics):
    sns.boxplot(data=quality, x='bucket', y=metric, hue='runtime', ax=axis)
    axis.set_title(metric)
plt.tight_layout()

## Efeito da revisão

In [ ]:
if deltas.empty:
    print('Sem deltas: informe as respostas canônicas para replay ou inclua runs closed-loop.')
else:
    display(deltas.groupby(['workload_profile', 'runtime', 'bucket']).agg(
        requests=('workload_request_id', 'count'),
        f1_delta_mean=('f1_delta', 'mean'),
        improvement_rate=('improved', 'mean'),
        regression_rate=('regressed', 'mean'),
    ).reset_index())